# 02 Data Cleaning

This notebook cleans the merged dataset and defines key variables
for subsequent analysis.

Input: merged_data.csv (from notebook 01)
Output: cleaned_data.csv (used in notebooks 03 and 04)

In [31]:
import pandas as pd

# Load merged dataset from notebook 01
df = pd.read_csv('../data/merged_data.csv')

In [32]:
# Check missing values before cleaning
# PFQ061D: 2766 missing due to NHANES questionnaire skip pattern
print(df.isnull().sum())
print("Total rows:", df.shape[0])

SEQN           0
RIDAGEYR       0
RIAGENDR       0
BMXBMI        99
PAD680        10
PAQ605         0
PFQ061D     2766
PFQ061H     2766
PFQ061I     2766
PFQ061M     2766
MCQ160A      268
dtype: int64
Total rows: 5533


In [33]:
print("Initial sample:", df.shape[0])

# Filter to working-age adults (20–65) to reduce age-related confounding
df = df[(df['RIDAGEYR'] >= 20) & (df['RIDAGEYR'] <= 65)]
print("After age filter:", df.shape[0])

# Remove invalid sedentary time values (9999 = don't know)
df = df[df['PAD680'] != 9999]
print("After sedentary cleaning:", df.shape[0])

# Remove missing and invalid physical activity values (7=refused, 9=don't know)
df = df[df['PAQ605'].notna()]
df = df[~df['PAQ605'].isin([7, 9])]
print("After activity cleaning:", df.shape[0])

# Remove missing and invalid functional limitation values (5=don't do, 7=refused, 9=don't know)
df = df[df['PFQ061D'].notna()]
df = df[~df['PFQ061D'].isin([5, 7, 9])]
print("After functional limitation cleaning:", df.shape[0])

# Complete-case analysis: imputation avoided as BMI is a primary predictor
df = df[df['BMXBMI'].notna()]
print("After BMI cleaning:", df.shape[0])

# Remove missing and invalid arthritis values (9=don't know)
df = df[df['MCQ160A'].notna()]
df = df[~df['MCQ160A'].isin([9])]
print("After arthritis cleaning:", df.shape[0])

# Binarize arthritis: 1=Yes, 0=No
# MCQ160A: 1=Yes, 2=No
df['arthritis'] = (df['MCQ160A'] == 1).astype(int)
print("Arthritis distribution:")
print(df['arthritis'].value_counts())

# Remove missing and invalid functional limitation values for all four items
# Valid codes: 1-4; exclude 5=don't do, 7=refused, 9=don't know
for col in ['PFQ061D', 'PFQ061H', 'PFQ061I', 'PFQ061M']:
    df = df[df[col].notna()]
    df = df[~df[col].isin([5, 7, 9])]
print("After functional items cleaning:", df.shape[0])

# Convert to hours; remove implausible values (extreme values likely reflect reporting errors)
df['sitting_hours'] = df['PAD680'] / 60
df = df[(df['sitting_hours'] >= 0.5) & (df['sitting_hours'] <= 18)]
print("Final sample:", df.shape[0])

Initial sample: 5533
After age filter: 3962
After sedentary cleaning: 3941
After activity cleaning: 3937
After functional limitation cleaning: 1424
After BMI cleaning: 1402
After arthritis cleaning: 1396
Arthritis distribution:
arthritis
0    796
1    600
Name: count, dtype: int64
After functional items cleaning: 1375
Final sample: 1347


The large reduction in sample size is primarily driven by missing 
PFQ061D responses, as functional limitation questions were only 
administered to eligible respondents under NHANES skip patterns.
Complete-case analysis may introduce selection bias; this is 
acknowledged as a study limitation.

In [34]:
# Recode each item: 1=0, 2=1, 3=2, 4=3
for col in ['PFQ061D', 'PFQ061H', 'PFQ061I', 'PFQ061M']:
    df[col + '_score'] = df[col].map({1: 0, 2: 1, 3: 2, 4: 3})

# Construct Functional Limitation Index (0-12)
df['FLI'] = df[['PFQ061D_score', 'PFQ061H_score', 
                 'PFQ061I_score', 'PFQ061M_score']].sum(axis=1)

# Binarize: 0 = no limitation (FLI=0), 1 = any limitation (FLI>=1)
# Primary outcome: FLI >= 1
df['functional_limitation'] = (df['FLI'] >= 1).astype(int)

# Secondary outcome: FLI >= 2 (for sensitivity analysis in notebook 04)
df['functional_limitation_strict'] = (df['FLI'] >= 2).astype(int)

print("Primary outcome (FLI>=1):", df['functional_limitation'].value_counts().to_dict())
print("Strict outcome (FLI>=2):", df['functional_limitation_strict'].value_counts().to_dict())

print("FLI distribution:")
print(df['FLI'].describe())
print("\nFunctional limitation (binary):")
print(df['functional_limitation'].value_counts())

Primary outcome (FLI>=1): {1: 877, 0: 470}
Strict outcome (FLI>=2): {0: 692, 1: 655}
FLI distribution:
count    1347.000000
mean        2.194506
std         2.400941
min         0.000000
25%         0.000000
50%         1.000000
75%         4.000000
max        12.000000
Name: FLI, dtype: float64

Functional limitation (binary):
functional_limitation
1    877
0    470
Name: count, dtype: int64


In [35]:
# Summary statistics of key continuous variables after cleaning
print("Final sample size:", df.shape[0])
print(df[['RIDAGEYR', 'BMXBMI', 'sitting_hours']].describe())

Final sample size: 1347
          RIDAGEYR       BMXBMI  sitting_hours
count  1347.000000  1347.000000    1347.000000
mean     53.014105    31.041871       5.592861
std      12.289074     8.115152       3.413729
min      20.000000    14.800000       0.500000
25%      46.000000    25.500000       3.000000
50%      59.000000    29.700000       5.000000
75%      62.000000    35.100000       8.000000
max      65.000000    68.200000      18.000000


In [36]:
print("Columns in cleaned dataset:")
print(df.columns.tolist())
df.to_csv('../data/cleaned_data.csv', index=False)
print("Cleaned data saved. Shape:", df.shape)

Columns in cleaned dataset:
['SEQN', 'RIDAGEYR', 'RIAGENDR', 'BMXBMI', 'PAD680', 'PAQ605', 'PFQ061D', 'PFQ061H', 'PFQ061I', 'PFQ061M', 'MCQ160A', 'arthritis', 'sitting_hours', 'PFQ061D_score', 'PFQ061H_score', 'PFQ061I_score', 'PFQ061M_score', 'FLI', 'functional_limitation', 'functional_limitation_strict']
Cleaned data saved. Shape: (1347, 20)
